# 示例
- 这个脚本模拟了 4 个进程（对应 4 张 GPU 卡），演示了如何先初始化全局通信环境，
- 再创建两个独立的子通信组（group1 包含 rank 0,1；group2 包含 rank 2,3），
- 并在各自的子组内进行独立的梯度同步（All-Reduce）

In [ ]:
import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp

def run_process(rank, world_size):
    # 1. 初始化全局进程组 (init_process_group)
    # 这是所有分布式通信的基础，让所有进程“互相认识”
    # 在单机测试时，我们使用 "gloo" 后端；实际多卡训练通常用 "nccl"
    # 是一个同步阻塞操作：等待所有进程调用init_process_group完成后，才会向下继续
    dist.init_process_group(backend="gloo", rank=rank, world_size=world_size)
    
    print(f"[Rank {rank}] 全局进程组初始化完成！")

    # 2. 创建多个子通信组 (new_group)
    # 核心铁律：所有进程（0, 1, 2, 3）都必须按完全相同的顺序执行 new_group
    # 即使某些进程不属于当前正在创建的组，也必须参与“建组握手”
    
    '''
    必须全员到齐：底层通信后端（如 NCCL）需要所有进程（rank 0, 1, 2, 3...）都到达这个“集合点”，
    共同完成通信 ID 的交换和握手，才能成功建立这个子组。
    '''
    # 所有进程同步创建第一个组（包含 rank 0 和 1）
    group1 = dist.new_group([0, 1])
    
    # 所有进程同步创建第二个组（包含 rank 2 和 3）
    group2 = dist.new_group([2, 3])

    # 3. 根据当前 rank 认领属于自己的通信组
    # 创建完成后，各进程判断自己属于哪个组，并保存对应的 group 对象
    my_group = None
    if rank in [0, 1]:
        my_group = group1
        print(f"[Rank {rank}] 成功认领 group1 (成员: 0, 1)")
    elif rank in [2, 3]:
        my_group = group2
        print(f"[Rank {rank}] 成功认领 group2 (成员: 2, 3)")

    # 4. 在子组内进行独立的集合通信 (All-Reduce)
    # 只有同一个组内的进程才会互相交换数据
    tensor = torch.ones(1) * (rank + 1) # 每个 rank 赋予不同的初始值
    print(f"[Rank {rank}] 通信前，我的张量值为: {tensor.item()}")
    
    if my_group is not None:
        # 组内求和：group1 的结果是 1+2=3；group2 的结果是 3+4=7
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM, group=my_group)
        print(f"[Rank {rank}] 组内 All-Reduce 通信后，我的张量值为: {tensor.item()}")

    # 销毁全局进程组，释放资源
    dist.destroy_process_group()

def main():
    world_size = 4
    # 使用 PyTorch 自带的多进程启动工具来模拟 4 个分布式进程
    # 启动 4 个进程，mp.spawn 会自动把 rank (0到3) 传给 run_process 函数
    mp.spawn(run_process, args=(world_size,), nprocs=world_size, join=True)

if __name__ == "__main__":
    # 设置多进程启动方法，避免在某些系统上报错
    mp.set_start_method("spawn", force=True)
    main()

## 预期输出结果
- 当你运行这段代码时，会看到类似以下的输出（由于多进程并发，不同 rank 的打印顺序可能会有交错，但逻辑是严格隔离的）

In [ ]:
[Rank 0] 全局进程组初始化完成！
[Rank 1] 全局进程组初始化完成！
[Rank 2] 全局进程组初始化完成！
[Rank 3] 全局进程组初始化完成！

[Rank 0] 成功认领 group1 (成员: 0, 1)
[Rank 1] 成功认领 group1 (成员: 0, 1)
[Rank 2] 成功认领 group2 (成员: 2, 3)
[Rank 3] 成功认领 group2 (成员: 2, 3)

[Rank 0] 通信前，我的张量值为: 1.0
[Rank 1] 通信前，我的张量值为: 2.0
[Rank 2] 通信前，我的张量值为: 3.0
[Rank 3] 通信前，我的张量值为: 4.0
    
[Rank 0] 组内 All-Reduce 通信后，我的张量值为: 3.0  <-- (1+2)
[Rank 1] 组内 All-Reduce 通信后，我的张量值为: 3.0  <-- (1+2)
[Rank 2] 组内 All-Reduce 通信后，我的张量值为: 7.0  <-- (3+4)
[Rank 3] 组内 All-Reduce 通信后，我的张量值为: 7.0  <-- (3+4)

## 代码关键点总结
- init_process_group：所有分布式训练的起点，必须在最前面调用，打通所有进程的基础通信。
- new_group 的全局同步性：<font color='red'>dist.new_group([0, 1]) 这行代码，rank 2 和 3 也必须跟着执行一遍，不能跳过，否则 rank 0 和 1 会无限阻塞等待。</font>
- group 参数的隔离作用：在调用 all_reduce 等通信算子时，传入 group=my_group，就能确保通信只发生在该子组内部，实现不同 GPU 小组之间的计算与通信隔离。